# GLORYS12 download + TEOS-10 preprocessing — Nordic Seas domain

Downloads monthly-mean GLORYS12 for the proposal Figure 1 domain and adds
TEOS-10 variables, so the figure notebook can apply exactly the same PSW
definition as Figure 5 of the Greenland Sea ice paper.

**Outputs** (one file per target month)

* raw: `RAW_DIR/glorys12_YYYYMM_NordicSeas.nc`
* processed: `PROC_DIR/glorys12_YYYYMM_NordicSeas_with_density.nc`
  (adds `sea_pressure`, `absolute_salinity`, `conservative_temperature`,
  `absolute_density`, same names and gsw chain as `preprocess_glorys12_v2`)

**Deliberate differences from `preprocess_glorys12_v2`**

* `delta_z`, `cell_area`, `cell_volume` are **not** written. They are not
  used by the PSW map, and `cell_volume` is a full 4-D array that would
  dominate file size for this domain.
* Only the variables needed for the map are downloaded
  (`so`, `thetao`, `siconc`); extend `VARIABLES` if needed later.
* Depth is truncated at `MAX_DEPTH`; the figure notebook asserts that no PSW
  cell reaches the deepest downloaded level.

**Credentials**: none are stored in this notebook. Run
`copernicusmarine login` once in a terminal (writes a local credentials
file); the toolbox then picks them up automatically.

In [3]:
import os
import glob
import calendar
import numpy as np
import xarray as xr
import gsw
import copernicusmarine

print(f"copernicusmarine {copernicusmarine.__version__}")

copernicusmarine 2.2.3


In [4]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Target months — (year, month). Figure 5b used a Feb–Mar mean; the proposal
# figure uses March 2025 only. Add (2025, 2) here AND in the figure notebook
# TARGET_MONTHS to reproduce the Feb–Mar convention.
TARGET_MONTHS = [(2025, 3)]

# Download domain = plot domain (60–85N, 40W–40E) + padding so the PSW field
# fills the curved map boundary.
LON_RANGE = [-45.0, 45.0]
LAT_RANGE = [ 58.0, 86.0]

# Depth: PSW (in-situ rho < 1027.7) is confined to the upper few hundred m;
# the figure notebook asserts the deepest level is never PSW.
MIN_DEPTH = 0.0
MAX_DEPTH = 1500.0

VARIABLES = ["so", "thetao", "siconc"]

# Monthly GLORYS12. Tried in order: the reprocessed ('my') stream first, then
# the interim ('myint') stream. The ID actually used is written into the
# processed file attributes and into the figure methods file.
DATASET_CANDIDATES = [
    "cmems_mod_glo_phy_my_0.083deg_P1M-m",
    "cmems_mod_glo_phy_myint_0.083deg_P1M-m",
]

RAW_DIR  = "../../glorys12_nordic_seas/raw/"
PROC_DIR = "../../glorys12_nordic_seas/with_density/"

# Skip logic is ALWAYS paired with explicit flags (stale-output lesson).
REDOWNLOAD = False   # True: delete and re-fetch existing raw files
REPROCESS  = False   # True: recompute processed files even if they exist

os.makedirs(RAW_DIR,  exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

def raw_name(y, m):  return f"glorys12_{y}{m:02d}_NordicSeas.nc"
def proc_name(y, m): return f"glorys12_{y}{m:02d}_NordicSeas_with_density.nc"

## 1. Download

Each target month is fetched as its own file. If a dataset ID does not
cover the requested month the toolbox raises; the next candidate is tried.

In [5]:
def download_month(year, month):
    out_file = os.path.join(RAW_DIR, raw_name(year, month))
    if os.path.exists(out_file):
        if not REDOWNLOAD:
            print(f"{raw_name(year, month)}: exists, skipped (REDOWNLOAD=False)")
            return out_file
        os.remove(out_file)

    last_day = calendar.monthrange(year, month)[1]
    start = f"{year}-{month:02d}-01 00:00:00"
    end   = f"{year}-{month:02d}-{last_day} 23:59:59"

    errors = {}
    for dataset_id in DATASET_CANDIDATES:
        print(f"{raw_name(year, month)}: trying {dataset_id}")
        try:
            copernicusmarine.subset(
                dataset_id        = dataset_id,
                variables         = VARIABLES,
                minimum_longitude = LON_RANGE[0],
                maximum_longitude = LON_RANGE[1],
                minimum_latitude  = LAT_RANGE[0],
                maximum_latitude  = LAT_RANGE[1],
                minimum_depth     = MIN_DEPTH,
                maximum_depth     = MAX_DEPTH,
                start_datetime    = start,
                end_datetime      = end,
                output_directory  = RAW_DIR,
                output_filename   = raw_name(year, month),
                file_format       = "netcdf",
            )
        except Exception as e:                      # noqa: BLE001
            errors[dataset_id] = repr(e)
            print(f"   failed: {e}")
            continue

        # Record provenance in the raw file itself
        with xr.open_dataset(out_file) as ds_chk:
            ds_chk = ds_chk.load()
        n_t = ds_chk.sizes.get("time", 0)
        if n_t != 1:
            raise RuntimeError(f"Expected 1 monthly time step, got {n_t}")
        ds_chk.attrs["glorys12_dataset_id"] = dataset_id
        tmp = out_file + ".tmp"
        ds_chk.to_netcdf(tmp)
        os.replace(tmp, out_file)
        print(f"   OK -> {out_file}  ({os.path.getsize(out_file)/1024**2:.1f} MB)")
        return out_file

    raise RuntimeError(f"No candidate dataset covered {year}-{month:02d}: {errors}")


raw_files = [download_month(y, m) for (y, m) in TARGET_MONTHS]

glorys12_202503_NordicSeas.nc: trying cmems_mod_glo_phy_my_0.083deg_P1M-m


WARNING - 2026-09-28T09:33:09Z - The following hosts are deprecated and will be removed in future versions: ['nrt.cmems-du.eu', 'my.cmems-du.eu']. Please update your netrc file to use the authentication system domain: auth.marine.copernicus.eu.
INFO - 2026-09-28T09:33:09Z - Selected dataset version: "202311"
INFO - 2026-09-28T09:33:09Z - Selected dataset part: "default"
WARNING - 2026-09-28T09:33:09Z - Some of your subset selection [0.0, 1500.0] for the depth dimension exceed the dataset coordinates [0.49402499198913574, 5727.9169921875]
INFO - 2026-09-28T09:33:15Z - Starting download. Please wait...


  0%|          | 0/72 [00:00<?, ?it/s]

INFO - 2026-09-28T09:33:34Z - Successfully downloaded to ../../glorys12_nordic_seas/raw/glorys12_202503_NordicSeas.nc


   OK -> ../../glorys12_nordic_seas/raw/glorys12_202503_NordicSeas.nc  (53.5 MB)


## 2. TEOS-10 preprocessing

Same calculation chain as `calculate_seawater_density()` in
`preprocess_glorys12_v2`:

`p = gsw.p_from_z(-depth, lat)` → `SA = gsw.SA_from_SP(so, p, lon, lat)` →
`CT = gsw.CT_from_pt(SA, thetao)` (GLORYS `thetao` is potential temperature) →
`rho = gsw.rho(SA, CT, p)` (in-situ density, the variable Figure 5 thresholds
at 1027.7 kg m⁻³).

The single monthly field is loaded into memory before the gsw calls, so no
dask is involved.

In [6]:
def add_teos10(ds):
    so     = ds["so"]
    thetao = ds["thetao"]

    lat_b = ds["latitude"].broadcast_like(so).transpose(*so.dims)
    lon_b = ds["longitude"].broadcast_like(so).transpose(*so.dims)
    z_b   = (-ds["depth"]).broadcast_like(so).transpose(*so.dims)

    p  = gsw.p_from_z(z_b, lat_b)
    SA = gsw.SA_from_SP(so, p, lon_b, lat_b)
    CT = gsw.CT_from_pt(SA, thetao)
    rho = gsw.rho(SA, CT, p)

    def _da(x, name, units, long_name):
        da = xr.DataArray(np.asarray(x), dims=so.dims, coords=so.coords)
        da = da.where(np.isfinite(so))          # keep land/seafloor NaN
        da.attrs = {"units": units, "long_name": long_name}
        return da.rename(name)

    ds["sea_pressure"]             = _da(p,   "sea_pressure", "dbar", "Sea Pressure")
    ds["absolute_salinity"]        = _da(SA,  "absolute_salinity", "g/kg",
                                         "Absolute Salinity (TEOS-10)")
    ds["conservative_temperature"] = _da(CT,  "conservative_temperature", "degC",
                                         "Conservative Temperature (TEOS-10)")
    ds["absolute_density"]         = _da(rho, "absolute_density", "kg/m^3",
                                         "Absolute (in-situ) Seawater Density (TEOS-10)")
    return ds


def preprocess_month(year, month):
    raw_file  = os.path.join(RAW_DIR,  raw_name(year, month))
    proc_file = os.path.join(PROC_DIR, proc_name(year, month))
    if os.path.exists(proc_file) and not REPROCESS:
        print(f"{proc_name(year, month)}: exists, skipped (REPROCESS=False)")
        return proc_file

    with xr.open_dataset(raw_file) as ds:
        ds = ds.load()

    missing = [v for v in ["so", "thetao", "siconc"] if v not in ds]
    if missing:
        raise KeyError(f"{raw_file} missing {missing}")

    ds = add_teos10(ds)
    ds.attrs["processing_history"] = (
        "TEOS-10 added: sea_pressure (gsw.p_from_z), absolute_salinity "
        "(gsw.SA_from_SP), conservative_temperature (gsw.CT_from_pt), "
        "absolute_density (gsw.rho, in-situ)."
    )
    ds.attrs["processing_date"] = str(np.datetime64("now"))

    enc = {v: {"zlib": True, "complevel": 4, "dtype": "float32"}
           for v in ["sea_pressure", "absolute_salinity",
                     "conservative_temperature", "absolute_density"]}
    ds.to_netcdf(proc_file, encoding=enc)
    print(f"{proc_name(year, month)}: written "
          f"({os.path.getsize(proc_file)/1024**2:.1f} MB), "
          f"dataset_id = {ds.attrs.get('glorys12_dataset_id', 'UNKNOWN')}")
    return proc_file


proc_files = [preprocess_month(y, m) for (y, m) in TARGET_MONTHS]

glorys12_202503_NordicSeas_with_density.nc: written (97.3 MB), dataset_id = cmems_mod_glo_phy_my_0.083deg_P1M-m


## 3. Sanity check

In [7]:
for f in proc_files:
    with xr.open_dataset(f) as ds:
        print(os.path.basename(f))
        print(f"  time      : {str(ds.time.values[0])[:10]}")
        print(f"  dataset   : {ds.attrs.get('glorys12_dataset_id', 'UNKNOWN')}")
        print(f"  lon       : {float(ds.longitude.min()):.2f} to {float(ds.longitude.max()):.2f}")
        print(f"  lat       : {float(ds.latitude.min()):.2f} to {float(ds.latitude.max()):.2f}")
        print(f"  depth     : {float(ds.depth.min()):.2f} to {float(ds.depth.max()):.1f} m "
              f"({ds.sizes['depth']} levels)")
        for v in ["so", "thetao", "absolute_density", "siconc"]:
            x = ds[v].values
            print(f"  {v:17s}: [{np.nanmin(x):.3f}, {np.nanmax(x):.3f}]  "
                  f"finite={int(np.isfinite(x).sum()):,}")

glorys12_202503_NordicSeas_with_density.nc
  time      : 2025-03-01
  dataset   : cmems_mod_glo_phy_my_0.083deg_P1M-m
  lon       : -45.00 to 45.00
  lat       : 58.00 to 86.00
  depth     : 0.49 to 1452.3 m (38 levels)
  so               : [2.120, 42.163]  finite=7,958,051
  thetao           : [-2.309, 10.719]  finite=7,958,051
  absolute_density : [1001.654, 1035.053]  finite=7,958,051
  siconc           : [0.000, 0.999]  finite=154,676


In [10]:
ds = xr.open_dataset('../../glorys12_nordic_seas/with_density/glorys12_202503_NordicSeas_with_density.nc')
hi = ds.so.isel(time=0).where(ds.so.isel(time=0) > 36, drop=True)
print(hi.count().item())
print(hi.stack(p=hi.dims).dropna("p").to_dataframe().head(20))

2269
                                   time     depth   latitude  longitude  \
depth    latitude  longitude                                              
0.494025 77.500000 15.083333 2025-03-01  0.494025  77.500000  15.083333   
                   15.166667 2025-03-01  0.494025  77.500000  15.166667   
                   21.500000 2025-03-01  0.494025  77.500000  21.500000   
                   21.583334 2025-03-01  0.494025  77.500000  21.583334   
                   21.666666 2025-03-01  0.494025  77.500000  21.666666   
                   22.000000 2025-03-01  0.494025  77.500000  22.000000   
                   22.083334 2025-03-01  0.494025  77.500000  22.083334   
                   22.750000 2025-03-01  0.494025  77.500000  22.750000   
         77.583336 15.000000 2025-03-01  0.494025  77.583336  15.000000   
                   15.083333 2025-03-01  0.494025  77.583336  15.083333   
                   15.166667 2025-03-01  0.494025  77.583336  15.166667   
                   1